# Steel temperature prediction

This notebook walks through the tested command-line workflow using only synthetic data.

The historical notebook used target-containing temperature aggregates. Historical scores are not valid quality claims. See the model card for evaluation boundaries.

In [ ]:
from pathlib import Path

import pandas as pd

from steel_model import FEATURES, TARGET, predict_file, read_frame, temporal_partitions, train_model

base = Path.cwd()
frame = read_frame(base / 'demo/batches.csv')
print(frame.head(3).to_string(index=False))


## Fixed feature contract

The model uses initial temperature and completed process-event aggregates. The final temperature is only a label. Batch identifiers and prediction timestamps are excluded from model input.

In [ ]:
print('Features:', FEATURES)
assert TARGET not in FEATURES
assert not any(name in FEATURES for name in ['temp_mean', 'temp_max', 'temp_min', 'batch_id', 'prediction_at'])


## Chronological evaluation

The latest 20% of distinct cutoff timestamps form the holdout. Tied timestamps stay together. Model selection uses three expanding folds on earlier records, with Ridge scaling inside a Pipeline.

In [ ]:
development, holdout = temporal_partitions(frame)
assert development.prediction_at.max() < holdout.prediction_at.min()
assert set(development.batch_id).isdisjoint(holdout.batch_id)
print('Development:', len(development), 'Holdout:', len(holdout))


In [ ]:
report = train_model(base / 'demo/batches.csv', base / 'output/notebook', synthetic=True)
comparison = pd.DataFrame({name: value['mae'] for name, value in report['cv'].items()}, index=['CV MAE']).T
print(comparison.sort_values('CV MAE').to_string())
print('Selected:', report['selected_model'])
print('Synthetic holdout:', report['holdout'])
assert report['business_target_met'] is None


## Saved-model prediction

Only load joblib files from trusted training runs, using the same dependency versions. These prediction rows are synthetic; no industrial quality claim is made.

In [ ]:
predictions = predict_file(base / 'output/notebook/model.joblib', base / 'demo/predict.csv',
                           base / 'output/notebook/new_predictions.csv')
print(predictions.to_string(index=False))


## Local-data run

With your own authorized raw files, run from the repository root:

```bash
python main.py prepare --raw-dir path/to/raw --output data/prepared.csv
python main.py train --csv data/prepared.csv --output-dir output/local
```

Raw records, real-data metrics and artifacts are not distributed. The public example does not assess the industrial MAE target.

See the data contract for event-cutoff assumptions. This is a retrospective late-process estimate, not a prediction at the beginning of treatment.